In [1]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TwinCheck'
REPO = '/content/Group-Project'
OUT = f'{DRIVE}/results/F5'
CKPT = f'{DRIVE}/models/F5'
!rm -rf {REPO}
!git clone -q --depth 1 https://github.com/himaallu/Group-Project.git {REPO}
!pip -q install onnx onnxruntime
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import hashlib
import io
import json
import os
import pickle
import platform
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps, JpegImagePlugin
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

os.chdir(REPO)
sys.path.insert(0, REPO)
from src.vision.detectors import CORRUPTIONS, build_model, corrupt, eval_transform, train_transform
from src.vision.gradcam import grad_cam, pick_examples, unnormalise
from src.vision.train_detector import metrics, set_seed

SEED, SIZE, POOL_SIZE = 42, 96, 500
EPOCHS, BATCH, LR, WORKERS = 3, 64, 1e-4, 2
ARCHS = ('resnet50', 'vit_b16')
SPLITS = ('train', 'validation', 'test')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FEAT = f'{DRIVE}/features/vision/genimage/clip_vitl14_debiased{SIZE}'
CACHE = f'{DRIVE}/data/processed/vision/genimage_debiased{SIZE}_jpeg.pkl'
os.makedirs(OUT, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

gen = pd.concat([pd.read_csv(f'data/processed/vision/genimage_{s}_manifest.csv') for s in SPLITS], ignore_index=True)
f3 = pd.read_csv(f'{FEAT}/embedding_manifest.csv')
assert len(gen) == len(f3) == 13000 and (gen['md5'].to_numpy() == f3['md5'].to_numpy()).all()
assert gen.groupby(['split', 'generator']).size().to_dict() == {
    ('test', 'Stable Diffusion v1.4'): 1000, ('test', 'real'): 1000,
    ('train', 'BigGAN'): 5000, ('train', 'real'): 4000,
    ('validation', 'BigGAN'): 1000, ('validation', 'real'): 1000}
gen['grayscale_added'] = f3['grayscale_added'].astype(bool).to_numpy()
Y = (gen['class_name'] == 'fake').astype(np.float32).to_numpy()
INDEX = {s: np.flatnonzero(gen['split'] == s) for s in SPLITS}
ROWS = {s: gen.iloc[INDEX[s]].reset_index(drop=True) for s in SPLITS}
assert os.path.exists(f'{FEAT}/embeddings_normalized.npy'), 'Run F3 first: its embeddings are needed for the CLIP probe'
assert DEVICE == 'cuda', 'Runtime > Change runtime type > T4 GPU'
print('images:', len(gen), '| greyscale added in F3:', int(gen['grayscale_added'].sum()), '| device:', torch.cuda.get_device_name(0))

images: 13000 | greyscale added in F3: 40 | device: Tesla T4


In [3]:
def jpeg_settings(path):
    with Image.open(path) as im:
        return [list(t) for t in im.quantization.values()], JpegImagePlugin.get_sampling(im)


def load_square(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        w, h = im.size
        s = min(w, h)
        left, top = (w - s) // 2, (h - s) // 2
        return np.asarray(im.crop((left, top, left + s, top + s)).resize((SIZE, SIZE), Image.Resampling.BICUBIC))


def debiased(i):
    a = load_square(f"{DRIVE}/{gen.at[i, 'raw_export_path']}")
    if gen.at[i, 'grayscale_added']:
        a = np.asarray(Image.fromarray(a).convert('L').convert('RGB'))
    tables, sampling = JPEG_POOL[int(hashlib.md5(gen.at[i, 'md5'].encode()).hexdigest()[:8], 16) % len(JPEG_POOL)]
    buf = io.BytesIO()
    Image.fromarray(a).save(buf, format='JPEG', qtables=tables, subsampling=sampling)
    return buf.getvalue()


if os.path.exists(CACHE):
    with open(CACHE, 'rb') as f:
        cache = pickle.load(f)
    print('Loaded the 96 px images from', CACHE)
else:
    pool_rows = gen[(gen['split'] == 'train') & (gen['class_name'] == 'real') & (gen['original_mode'] == 'RGB')].sample(POOL_SIZE, random_state=SEED)
    with ThreadPoolExecutor(32) as pool:
        settings = list(tqdm(pool.map(lambda p: jpeg_settings(f'{DRIVE}/{p}'), pool_rows['raw_export_path']), total=len(pool_rows), desc='Reading JPEG settings'))
    JPEG_POOL = [s for s in settings if len(s[0]) >= 2 and s[1] in (0, 1, 2)]
    with ThreadPoolExecutor(32) as pool:
        blobs = list(tqdm(pool.map(debiased, range(len(gen))), total=len(gen), desc='Making the 96 px images'))
    cache = {'md5': gen['md5'].tolist(), 'jpeg': blobs, 'jpeg_settings': len(JPEG_POOL)}
    with open(CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print('Saved the 96 px images to', CACHE)
assert cache['md5'] == gen['md5'].tolist()
BLOBS = cache['jpeg']
print('images:', len(BLOBS), '| JPEG settings from real photos:', cache['jpeg_settings'], '| size in memory:', round(sum(map(len, BLOBS)) / 1e6, 1), 'MB')

Reading JPEG settings:   0%|          | 0/500 [00:00<?, ?it/s]

Making the 96 px images:   0%|          | 0/13000 [00:00<?, ?it/s]

Saved the 96 px images to /content/drive/MyDrive/TwinCheck/data/processed/vision/genimage_debiased96_jpeg.pkl
images: 13000 | JPEG settings from real photos: 500 | size in memory: 92.2 MB


In [4]:
class Images(Dataset):
    def __init__(self, split, transform, corruption='clean'):
        self.idx, self.transform, self.corruption = INDEX[split], transform, corruption

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        image = corrupt(Image.open(io.BytesIO(BLOBS[i])).convert('RGB'), self.corruption)
        return self.transform(image), torch.tensor(Y[i]), i


def loader(split, transform, corruption='clean', shuffle=False):
    return DataLoader(Images(split, transform, corruption), batch_size=BATCH, shuffle=shuffle, num_workers=WORKERS,
                      pin_memory=True, generator=torch.Generator().manual_seed(SEED) if shuffle else None)


@torch.inference_mode()
def predict(model, dl, desc):
    model.eval()
    probs = []
    for x, _, _ in tqdm(dl, desc=desc, leave=False):
        with torch.autocast('cuda', dtype=torch.float16):
            probs.append(torch.sigmoid(model(x.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)).squeeze(1).float()).cpu().numpy())
    return np.concatenate(probs)


def load_detector(arch):
    model = build_model(arch, pretrained=False).to(DEVICE, memory_format=torch.channels_last)
    model.load_state_dict(torch.load(f'{CKPT}/{arch}_s{SEED}.pt', map_location=DEVICE)['model'])
    return model.eval()


def train(arch):
    set_seed(SEED)
    path = f'{CKPT}/{arch}_s{SEED}.pt'
    model = build_model(arch).to(DEVICE, memory_format=torch.channels_last)
    params = [p for p in model.parameters() if p.requires_grad]
    train_dl, val_dl = loader('train', train_transform(), shuffle=True), loader('validation', eval_transform())
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_dl))
    scaler = torch.amp.GradScaler()
    loss_fn = nn.BCEWithLogitsLoss()
    start, history = 0, []
    if os.path.exists(path):
        state = torch.load(path, map_location=DEVICE)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        start, history = state['epoch'] + 1, state['history']
        print(f'{arch}: continuing after epoch {start}')
    for epoch in range(start, EPOCHS):
        model.train()
        total = seen = 0
        for x, y, _ in tqdm(train_dl, desc=f'{arch} epoch {epoch + 1}/{EPOCHS}'):
            x = x.to(DEVICE, non_blocking=True).contiguous(memory_format=torch.channels_last)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = loss_fn(model(x).squeeze(1), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total, seen = total + loss.item() * len(y), seen + len(y)
        val = metrics(predict(model, val_dl, 'validation'), Y[INDEX['validation']], ROWS['validation'])
        history.append({'epoch': epoch + 1, 'train_loss': total / seen, 'val_accuracy': val['accuracy']})
        print(history[-1])
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                    'epoch': epoch, 'history': history}, path)
    result = {'seed': SEED, 'arch': arch, 'split': 'F1 (train/val real + BigGAN, test real + SD v1.4)',
              'input': f'F3 preprocessing: centre crop, {SIZE} px, JPEG with real-photo settings, greyscale share matched',
              'epochs': EPOCHS, 'lr': LR, 'batch': BATCH, 'unfrozen': 'last 4 blocks + head',
              'augmentation': 'rrc0.6, hflip, jpeg60-95, blur p0.3', 'mixed_precision': 'fp16',
              'n_train': len(INDEX['train']), 'n_val': len(INDEX['validation']), 'n_test': len(INDEX['test']), 'history': history}
    for split in ('validation', 'test'):
        result[split] = metrics(predict(model, loader(split, eval_transform()), split), Y[INDEX[split]], ROWS[split])
    with open(f'{OUT}/train_{arch}_s{SEED}.json', 'w') as f:
        json.dump(result, f, indent=2)
    return result


training = {}
for arch in ARCHS:
    training[arch] = train(arch)
    torch.cuda.empty_cache()
print(pd.DataFrame([{'model': a, 'split': s, **{k: training[a][s][k] for k in ('accuracy', 'precision', 'recall', 'f1', 'auroc')}}
                    for a in ARCHS for s in ('validation', 'test')]).round(3).to_string(index=False))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 142MB/s]


resnet50 epoch 1/3:   0%|          | 0/141 [00:00<?, ?it/s]

validation:   0%|          | 0/32 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.30082741808891295, 'val_accuracy': 0.802}


resnet50 epoch 2/3:   0%|          | 0/141 [00:00<?, ?it/s]

... (17 outputs left out)

{'epoch': 3, 'train_loss': 0.0066589631003638105, 'val_accuracy': 0.9915}


validation:   0%|          | 0/32 [00:00<?, ?it/s]

test:   0%|          | 0/32 [00:00<?, ?it/s]

   model      split  accuracy  precision  recall    f1  auroc
resnet50 validation     0.842      0.997   0.687 0.813  0.982
resnet50       test     0.504      0.765   0.013 0.026  0.610
 vit_b16 validation     0.992      0.992   0.991 0.991  1.000
 vit_b16       test     0.555      0.887   0.126 0.221  0.754


In [5]:
rows = []
for arch in ARCHS:
    model = load_detector(arch)
    for split in ('validation', 'test'):
        for kind in CORRUPTIONS:
            p_ai = predict(model, loader(split, eval_transform(), kind), f'{arch} {split} {kind}')
            rows.append({'model': arch, 'seed': SEED, 'split': split, 'corruption': kind, **metrics(p_ai, Y[INDEX[split]], ROWS[split])})
    del model
    torch.cuda.empty_cache()
detector_rows = pd.DataFrame(rows)
print(detector_rows[['model', 'split', 'corruption', 'accuracy', 'recall', 'auroc']].round(3).to_string(index=False))

resnet50 validation clean:   0%|          | 0/32 [00:00<?, ?it/s]

resnet50 validation jpeg50:   0%|          | 0/32 [00:00<?, ?it/s]

resnet50 validation jpeg70:   0%|          | 0/32 [00:00<?, ?it/s]

resnet50 validation crop20:   0%|          | 0/32 [00:00<?, ?it/s]

resnet50 test clean:   0%|          | 0/32 [00:00<?, ?it/s]

resnet50 test jpeg50:   0%|          | 0/32 [00:00<?, ?it/s]

... (7 outputs left out)

vit_b16 test jpeg50:   0%|          | 0/32 [00:00<?, ?it/s]

vit_b16 test jpeg70:   0%|          | 0/32 [00:00<?, ?it/s]

vit_b16 test crop20:   0%|          | 0/32 [00:00<?, ?it/s]

   model      split corruption  accuracy  recall  auroc
resnet50 validation      clean     0.842   0.687  0.982
resnet50 validation     jpeg50     0.822   0.658  0.965
resnet50 validation     jpeg70     0.846   0.696  0.982
resnet50 validation     crop20     0.874   0.753  0.985
resnet50       test      clean     0.504   0.013  0.610
resnet50       test     jpeg50     0.506   0.031  0.599
resnet50       test     jpeg70     0.511   0.027  0.626
resnet50       test     crop20     0.506   0.027  0.582
 vit_b16 validation      clean     0.992   0.991  1.000
 vit_b16 validation     jpeg50     0.977   0.987  0.997
 vit_b16 validation     jpeg70     0.985   0.991  0.998
 vit_b16 validation     crop20     0.988   0.992  0.999
 vit_b16       test      clean     0.555   0.126  0.754
 vit_b16       test     jpeg50     0.566   0.176  0.724
 vit_b16       test     jpeg70     0.563   0.156  0.745
 vit_b16       test     crop20     0.559   0.140  0.736


In [6]:
from sklearn.linear_model import LogisticRegression
from torchvision import transforms
from transformers import CLIPVisionModelWithProjection

X_f3 = np.load(f'{FEAT}/embeddings_normalized.npy').astype(np.float32)
probe = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_f3[INDEX['train']], Y[INDEX['train']])
clip = CLIPVisionModelWithProjection.from_pretrained('openai/clip-vit-large-patch14', torch_dtype=torch.float16).to(DEVICE).eval()
clip_tensor = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]),
])


@torch.inference_mode()
def clip_embed(split, kind):
    out = []
    for x, _, _ in tqdm(DataLoader(Images(split, clip_tensor, kind), batch_size=128, num_workers=WORKERS), desc=f'CLIP {split} {kind}', leave=False):
        out.append(torch.nn.functional.normalize(clip(pixel_values=x.to(DEVICE, torch.float16)).image_embeds.float(), dim=1).cpu().numpy())
    return np.concatenate(out)


clip_rows = []
for split in ('validation', 'test'):
    for kind in CORRUPTIONS:
        E = clip_embed(split, kind)
        if kind == 'clean':
            same = float((E * X_f3[INDEX[split]]).sum(1).mean())
            print(f'{split}: mean cosine with the F3 embeddings {same:.4f}')
            assert same > 0.99
        clip_rows.append({'model': 'clip_probe', 'seed': SEED, 'split': split, 'corruption': kind,
                          **metrics(probe.predict_proba(E)[:, 1], Y[INDEX[split]], ROWS[split])})
del clip
torch.cuda.empty_cache()
cnn_vit = pd.concat([detector_rows, pd.DataFrame(clip_rows)], ignore_index=True)
cnn_vit.to_csv(f'{OUT}/cnn_vit.csv', index=False)
table = cnn_vit.pivot_table(index=['split', 'model'], columns='corruption', values='accuracy')[list(CORRUPTIONS)]
print('accuracy')
print(table.round(3).to_string())
print('AUROC')
print(cnn_vit.pivot_table(index=['split', 'model'], columns='corruption', values='auroc')[list(CORRUPTIONS)].round(3).to_string())

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.embeddings.token_embedding.weight                 | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{

CLIP validation clean:   0%|          | 0/16 [00:00<?, ?it/s]

... (6 outputs left out)

CLIP test jpeg50:   0%|          | 0/16 [00:00<?, ?it/s]

CLIP test jpeg70:   0%|          | 0/16 [00:00<?, ?it/s]

CLIP test crop20:   0%|          | 0/16 [00:00<?, ?it/s]

accuracy
corruption             clean  jpeg50  jpeg70  crop20
split      model                                    
test       clip_probe  0.542   0.525   0.526   0.547
           resnet50    0.504   0.506   0.511   0.506
           vit_b16     0.555   0.566   0.563   0.559
validation clip_probe  0.958   0.908   0.910   0.946
           resnet50    0.842   0.822   0.846   0.874
           vit_b16     0.992   0.977   0.985   0.988
AUROC
corruption             clean  jpeg50  jpeg70  crop20
split      model                                    
test       clip_probe  0.620   0.578   0.577   0.598
           resnet50    0.610   0.599   0.626   0.582
           vit_b16     0.754   0.724   0.745   0.736
validation clip_probe  0.992   0.962   0.971   0.985
           resnet50    0.982   0.965   0.982   0.985
           vit_b16     1.000   0.997   0.998   0.999


In [7]:
import matplotlib.pyplot as plt

examples = pick_examples(SEED)
position = {m: i for i, m in enumerate(gen['md5'])}
data = [eval_transform()(Image.open(io.BytesIO(BLOBS[position[m]])).convert('RGB')) for m in examples['image_id']]
x = torch.stack(data).to(DEVICE)
for arch in ARCHS:
    model = build_model(arch, pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(f'{CKPT}/{arch}_s{SEED}.pt', map_location=DEVICE)['model'])
    cams, p_ai = grad_cam(model, arch, x)
    fig, axes = plt.subplots(2, 4, figsize=(12, 7.2))
    for i, ax in enumerate(axes.flat):
        ax.imshow(unnormalise(x[i]))
        ax.imshow(cams[i], cmap='jet', alpha=0.4)
        ax.set_title(f"{examples['generator'][i]} ({examples['split'][i]})\nP(AI) = {p_ai[i]:.2f}", fontsize=9)
        ax.axis('off')
    fig.suptitle(f'Grad-CAM, {arch}')
    fig.tight_layout()
    fig.savefig(f'{OUT}/fig_gradcam_{arch}.png', dpi=300)
    plt.show()
    plt.close(fig)
    examples[['image_id', 'split', 'generator', 'class_name']].assign(p_ai=p_ai).to_csv(f'{OUT}/gradcam_{arch}.csv', index=False)
    print(arch, dict(zip(examples['generator'] + ' ' + examples['image_id'].str[:6], p_ai.round(3))))
    del model
    torch.cuda.empty_cache()

resnet50 {'real 81a38d': np.float32(0.001), 'real 5030d1': np.float32(0.001), 'real 3773d8': np.float32(0.002), 'Stable Diffusion v1.4 017684': np.float32(0.001), 'Stable Diffusion v1.4 225ed8': np.float32(0.005), 'Stable Diffusion v1.4 ddb0cc': np.float32(0.01), 'BigGAN e6618d': np.float32(0.959), 'BigGAN 9eabf0': np.float32(0.07)}
vit_b16 {'real 81a38d': np.float32(0.0), 'real 5030d1': np.float32(0.0), 'real 3773d8': np.float32(0.0), 'Stable Diffusion v1.4 017684': np.float32(0.001), 'Stable Diffusion v1.4 225ed8': np.float32(0.198), 'Stable Diffusion v1.4 ddb0cc': np.float32(0.0), 'BigGAN e6618d': np.float32(1.0), 'BigGAN 9eabf0': np.float32(0.995)}


In [8]:
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic
from sklearn.metrics import accuracy_score, roc_auc_score

torch.backends.mha.set_fastpath_enabled(False)
WARMUP, RUNS, CHECK_N = 10, 100, 500
check_idx = np.sort(ROWS['test'].groupby('class_name', group_keys=False).sample(CHECK_N // 2, random_state=SEED).index.to_numpy())
check_y = Y[INDEX['test']][check_idx]


def check_batches():
    for s in range(0, len(check_idx), 50):
        yield np.stack([eval_transform()(Image.open(io.BytesIO(BLOBS[INDEX['test'][k]])).convert('RGB')).numpy() for k in check_idx[s:s + 50]])


def time_ms(fn):
    for _ in range(WARMUP):
        fn()
    times = []
    for _ in range(RUNS):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter() - start) * 1000)
    return np.array(times)


def onnx_check(session, desc):
    logits = np.concatenate([session.run(None, {'image': b})[0][:, 0] for b in tqdm(check_batches(), total=CHECK_N // 50, desc=desc, leave=False)])
    return 1 / (1 + np.exp(-logits))


latency, int8 = [], []
one = np.random.default_rng(0).standard_normal((1, 3, 224, 224)).astype(np.float32)
check = torch.randn(4, 3, 224, 224)
for arch in ARCHS:
    model = build_model(arch, pretrained=False).eval()
    model.load_state_dict(torch.load(f'{CKPT}/{arch}_s{SEED}.pt', map_location='cpu')['model'])
    fp32_path, int8_path = f'{CKPT}/{arch}_s{SEED}.onnx', f'{CKPT}/{arch}_s{SEED}_int8.onnx'
    torch.onnx.export(model, torch.randn(1, 3, 224, 224), fp32_path, opset_version=17, input_names=['image'], output_names=['logit'],
                      dynamic_axes={'image': {0: 'batch'}, 'logit': {0: 'batch'}}, dynamo=False)
    quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QUInt8)
    with torch.no_grad():
        reference = model(check).numpy()
    gpu_model, gx = model.cuda(), torch.from_numpy(one).cuda()
    with torch.no_grad():
        torch_p = torch.sigmoid(torch.cat([gpu_model(torch.from_numpy(b).cuda()) for b in check_batches()])[:, 0]).cpu().numpy()

    def gpu_call():
        with torch.no_grad():
            gpu_model(gx)
        torch.cuda.synchronize()

    gpu_ms = float(time_ms(gpu_call).mean())
    for precision, path in (('fp32', fp32_path), ('int8', int8_path)):
        session = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
        diff = float(np.abs(session.run(None, {'image': check.numpy()})[0] - reference).max())
        cpu_ms = time_ms(lambda: session.run(None, {'image': one}))
        p = onnx_check(session, f'{arch} {precision} ONNX')
        row = {'arch': arch, 'onnx_mb': round(os.path.getsize(path) / 1e6, 1), 'max_abs_diff': diff,
               'ort_cpu_ms_mean': float(cpu_ms.mean()), 'ort_cpu_ms_p50': float(np.median(cpu_ms))}
        quality = {'test_images': CHECK_N, 'test_accuracy': accuracy_score(check_y, p >= 0.5), 'test_auroc': roc_auc_score(check_y, p),
                   'same_decision_as_pytorch': float(((p >= 0.5) == (torch_p >= 0.5)).mean())}
        if precision == 'fp32':
            latency.append({**row, 'torch_gpu_ms_mean': gpu_ms, 'cpu': platform.processor() or platform.machine(),
                            'cpu_threads': os.cpu_count(), 'gpu': torch.cuda.get_device_name(0)})
        int8.append({'arch': arch, 'precision': precision, **row, **quality})
        print(int8[-1])
    del model, gpu_model
    torch.cuda.empty_cache()
pd.DataFrame(latency).to_csv(f'{OUT}/onnx_latency.csv', index=False)
pd.DataFrame(int8).to_csv(f'{OUT}/onnx_int8.csv', index=False)
print(pd.DataFrame(int8).round(4).to_string(index=False))

/tmp/ipykernel_494/3405908761.py:39: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.randn(1, 3, 224, 224), fp32_path, opset_version=17, input_names=['image'], output_names=['logit'],


resnet50 fp32 ONNX:   0%|          | 0/10 [00:00<?, ?it/s]

{'arch': 'resnet50', 'precision': 'fp32', 'onnx_mb': 94.0, 'max_abs_diff': 8.0108642578125e-05, 'ort_cpu_ms_mean': 96.19000189999497, 'ort_cpu_ms_p50': 85.60177099991506, 'test_images': 500, 'test_accuracy': 0.512, 'test_auroc': np.float64(0.6255839999999999), 'same_decision_as_pytorch': 1.0}


resnet50 int8 ONNX:   0%|          | 0/10 [00:00<?, ?it/s]

{'arch': 'resnet50', 'precision': 'int8', 'onnx_mb': 23.7, 'max_abs_diff': 10.683120727539062, 'ort_cpu_ms_mean': 141.40846810000767, 'ort_cpu_ms_p50': 128.8605644999734, 'test_images': 500, 'test_accuracy': 0.502, 'test_auroc': np.float64(0.8260480000000001), 'same_decision_as_pytorch': 0.99}


/tmp/ipykernel_494/3405908761.py:39: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.randn(1, 3, 224, 224), fp32_path, opset_version=17, input_names=['image'], output_names=['logit'],
/usr/local/lib/python3.13/dist-packages/torch/__init__.py:2675: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not condition:


vit_b16 fp32 ONNX:   0%|          | 0/10 [00:00<?, ?it/s]

{'arch': 'vit_b16', 'precision': 'fp32', 'onnx_mb': 343.6, 'max_abs_diff': 1.3113021850585938e-06, 'ort_cpu_ms_mean': 481.6913923500033, 'ort_cpu_ms_p50': 423.2429505000255, 'test_images': 500, 'test_accuracy': 0.566, 'test_auroc': np.float64(0.770816), 'same_decision_as_pytorch': 1.0}


vit_b16 int8 ONNX:   0%|          | 0/10 [00:00<?, ?it/s]

{'arch': 'vit_b16', 'precision': 'int8', 'onnx_mb': 87.3, 'max_abs_diff': 0.20382440090179443, 'ort_cpu_ms_mean': 432.67823472998765, 'ort_cpu_ms_p50': 398.7988065000536, 'test_images': 500, 'test_accuracy': 0.576, 'test_auroc': np.float64(0.769552), 'same_decision_as_pytorch': 0.982}
    arch precision  onnx_mb  max_abs_diff  ort_cpu_ms_mean  ort_cpu_ms_p50  test_images  test_accuracy  test_auroc  same_decision_as_pytorch
resnet50      fp32     94.0        0.0001          96.1900         85.6018          500          0.512      0.6256                     1.000
resnet50      int8     23.7       10.6831         141.4085        128.8606          500          0.502      0.8260                     0.990
 vit_b16      fp32    343.6        0.0000         481.6914        423.2430          500          0.566      0.7708                     1.000
 vit_b16      int8     87.3        0.2038         432.6782        398.7988          500          0.576      0.7696                     0.982


In [9]:
test = cnn_vit[(cnn_vit['split'] == 'test') & (cnn_vit['corruption'] == 'clean')]
print('Unseen generator (real + SD v1.4), clean')
print(test[['model', 'accuracy', 'precision', 'recall', 'f1', 'auroc']].round(3).to_string(index=False))
for f in sorted(os.listdir(OUT)):
    print(f)
print(sorted(os.listdir(CKPT)))

Unseen generator (real + SD v1.4), clean
     model  accuracy  precision  recall    f1  auroc
  resnet50     0.504      0.765   0.013 0.026  0.610
   vit_b16     0.555      0.887   0.126 0.221  0.754
clip_probe     0.542      0.684   0.156 0.254  0.620
cnn_vit.csv
fig_gradcam_resnet50.png
fig_gradcam_vit_b16.png
gradcam_resnet50.csv
gradcam_vit_b16.csv
onnx_int8.csv
onnx_latency.csv
train_resnet50_s42.json
train_vit_b16_s42.json
['resnet50_s42.onnx', 'resnet50_s42.pt', 'resnet50_s42_int8.onnx', 'vit_b16_s42.onnx', 'vit_b16_s42.pt', 'vit_b16_s42_int8.onnx']
